# 🎨 AbiArt — Kaggle GPU Backend (Automatic1111 SD WebUI API)

### ⚡ Configuration requise dans Kaggle (panneau à droite)
1. **Accelerator** → `GPU T4 x2` (ou `P100`)
2. **Internet** → `ON`
3. Cliquez sur **Run All** (ou exécutez les cellules 1 à 5 dans l'ordre).
4. À la fin, copiez l'URL Cloudflare affichée (`https://xxxx.trycloudflare.com`) et collez-la dans AbiArt → Backend Kaggle → Connect.

### 🔧 Pourquoi cette version ?
Kaggle tourne maintenant sous **Python 3.13** avec **transformers 5.x**. Automatic1111 (v1.10.1) a été écrit pour
**Python 3.10 + torch 2.1.2 + transformers 4.30.2** : le chargement du modèle plantait avec
`RuntimeError: cannot schedule new futures after interpreter shutdown`.
Ce notebook crée donc un **environnement Python 3.10 isolé** (via `uv`) uniquement pour A1111, sans toucher à l'environnement Kaggle.

📥 La cellule 5 installe aussi une petite extension qui permet au site AbiArt de télécharger des modèles Civitai / Hugging Face directement sur Kaggle.

🧑‍🎨 La cellule 2 installe aussi **ADetailer** (correction automatique des visages et des mains) : +2 à 3 min la première fois.

⏱️ Première exécution : ~10 à 15 min (installation + téléchargement du modèle de 6,9 Go). Ensuite, la cellule 5 démarre en 1-2 min.

### 💾 Sauvegarde des modèles (dépôt Hugging Face privé) — évite de tout retélécharger à chaque redémarrage
Kaggle efface le disque à chaque nouvelle session. Pour garder vos checkpoints / LoRAs / VAE :
1. Sur [huggingface.co](https://huggingface.co/new) : créez un dépôt **modèle privé** (ex. `monpseudo/abiart-models`).
2. Settings → Access Tokens : créez un jeton avec le droit d'**écriture** sur ce dépôt.
3. Dans ce notebook Kaggle : **Add-ons → Secrets**, ajoutez `HF_TOKEN` (le jeton) et `HF_REPO` (ex. `monpseudo/abiart-models`), puis activez-les pour ce notebook.

Ensuite : chaque modèle téléchargé depuis le site AbiArt est envoyé automatiquement (en arrière-plan) dans le dépôt, et la **cellule 4 restaure tout** au démarrage d'une nouvelle session. Pour supprimer un modèle sauvegardé, supprimez son fichier dans le dépôt Hugging Face.

### 🚀 Deux GPU (T4 x2)
La cellule 5 lance **une instance A1111 par GPU**. Une image seule n'est pas calculée plus vite, mais **deux générations tournent en même temps** (deux utilisateurs, ou deux images d'un même lot), et chaque GPU peut garder un checkpoint différent chargé. Si la 2e instance ne démarre pas, le backend continue sur un seul GPU.


In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 1 — VÉRIFICATION DU GPU
# ══════════════════════════════════════════════════
import subprocess, shutil

has_gpu = False
if shutil.which('nvidia-smi'):
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                           capture_output=True, text=True)
        if r.returncode == 0 and r.stdout.strip():
            has_gpu = True
            print('✓ GPU actif et détecté :')
            print('  ' + r.stdout.strip().replace('\n', '\n  '))
    except Exception:
        pass

if not has_gpu:
    print('❌ ERREUR CRITIQUE : le GPU n\'est pas activé sur Kaggle !')
    print('    → Volet de droite → "Session options" → "Accelerator"')
    print('    → Choisissez "GPU T4 x2" (ou "GPU P100")')
    print('    → Vérifiez que "Internet" est sur "On"')
    print('    → Redémarrez la session.')
    raise RuntimeError('Veuillez activer le GPU dans le panneau de droite avant de continuer.')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 2 — INSTALLATION (Python 3.10 isolé + A1111 v1.10.1)
# ══════════════════════════════════════════════════
import os, sys, shutil, subprocess

def sh(cmd, cwd=None, env=None):
    """Exécute une commande shell et affiche la sortie en direct dans la cellule."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f'Commande échouée (code {p.returncode}) : {cmd}')

# 0. Choix du dossier de travail (celui qui a le plus d'espace libre ; /kaggle/working est limité à 20 Go)
cands = [d for d in ('/kaggle/temp', '/tmp') if os.path.isdir(d) and os.access(d, os.W_OK)] or ['/kaggle/working']
BASE = max(cands, key=lambda d: shutil.disk_usage(d).free) + '/abiart'
os.makedirs(BASE, exist_ok=True)
with open('/tmp/abiart_base.txt', 'w') as f:
    f.write(BASE)

WEBUI = f'{BASE}/stable-diffusion-webui'
VENV  = f'{BASE}/venv'
PY    = f'{VENV}/bin/python'
UV    = f'"{sys.executable}" -m uv'
print(f'Dossier de travail : {BASE}  ({shutil.disk_usage(BASE).free / 1024**3:.1f} Go libres)')

# 1. uv (installeur rapide, capable de télécharger Python 3.10)
print('\n[1/5] Installation de uv...')
sh(f'"{sys.executable}" -m pip install -q -U uv')

# 2. Environnement virtuel Python 3.10
print('\n[2/5] Création de l\'environnement Python 3.10...')
if not os.path.exists(PY):
    sh(f'{UV} venv --seed --python 3.10 "{VENV}"')
sh(f'"{PY}" --version')

# 3. Cloner Automatic1111 (version stable v1.10.1)
print('\n[3/5] Clonage d\'Automatic1111 v1.10.1...')
if not os.path.exists(f'{WEBUI}/launch.py'):
    shutil.rmtree(WEBUI, ignore_errors=True)
    sh(f'git clone -q --depth 1 --branch v1.10.1 https://github.com/AUTOMATIC1111/stable-diffusion-webui.git "{WEBUI}"')
else:
    print('  ✓ déjà présent')

# 4. Dépendances : exactement celles qu'A1111 attend (requirements_versions.txt) + torch 2.1.2
print('\n[4/5] Installation des dépendances Python (torch 2.1.2, transformers 4.30.2, gradio 3.41.2...)')
marker = f'{VENV}/.abiart_deps_ok'
if not os.path.exists(marker):
    lines = []
    for l in open(f'{WEBUI}/requirements_versions.txt').read().splitlines():
        l = l.strip()
        if l and l != 'torch':
            lines.append(l)
    lines += ['torch==2.1.2', 'torchvision==0.16.2', 'ftfy', 'regex']
    req = f'{BASE}/requirements_kaggle.txt'
    with open(req, 'w') as f:
        f.write('\n'.join(lines) + '\n')
    sh(f'{UV} pip install --python "{PY}" -r "{req}"')
    # CLIP : doit être compilé avec le setuptools épinglé (69.5.1) qui contient encore pkg_resources
    sh(f'{UV} pip install --python "{PY}" --no-build-isolation --no-deps '
       f'https://github.com/openai/CLIP/archive/d50d76daa670286dd6cacf3bcd80b5e4823fc8e1.zip')
    open(marker, 'w').write('ok')
else:
    print('  ✓ déjà installées')

# 4b. ADetailer : correction automatique des visages / mains (extension A1111 + ultralytics).
#     Facultatif : en cas d'échec, le reste du backend fonctionne sans ADetailer.
print('\n[4b/5] ADetailer (visages / mains)...')
try:
    AD_DIR = f'{WEBUI}/extensions/adetailer'
    if not os.path.exists(f'{AD_DIR}/scripts'):
        shutil.rmtree(AD_DIR, ignore_errors=True)
        sh(f'git clone -q --depth 1 --branch v24.11.1 https://github.com/Bing-su/adetailer.git "{AD_DIR}"')
    ad_marker = f'{VENV}/.abiart_adetailer_ok'
    if not os.path.exists(ad_marker):
        # -c : les versions épinglées d'A1111 (numpy, pillow, protobuf...) ne sont pas modifiées.
        # mediapipe n'est pas installé volontairement (facultatif, et incompatible avec le protobuf d'A1111).
        req_file = f'{BASE}/requirements_kaggle.txt'
        constraint = f'-c "{req_file}" ' if os.path.exists(req_file) else ''
        sh(f'{UV} pip install --python "{PY}" {constraint}"ultralytics==8.3.40" "rich>=13"')
        open(ad_marker, 'w').write('ok')
    else:
        print('  ✓ déjà installé')
except Exception as e:
    print(f'  ⚠️ ADetailer non installé ({e}) : le reste du backend fonctionne sans.')

# 5. Sous-dépôts d'A1111 (stable-diffusion, generative-models, k-diffusion, BLIP, assets)
print('\n[5/5] Clonage des sous-dépôts A1111...')
env = os.environ.copy()
for k in ('PYTHONPATH', 'PYTHONHOME', 'VIRTUAL_ENV'):
    env.pop(k, None)
env['STABLE_DIFFUSION_REPO'] = 'https://github.com/w-e-w/stablediffusion.git'  # dépôt Stability-AI d'origine retiré
env['PATH'] = f'{VENV}/bin:' + env.get('PATH', '')
env['MPLBACKEND'] = 'Agg'
sh(f'"{PY}" launch.py --exit --skip-torch-cuda-test --skip-install', cwd=WEBUI, env=env)

total, used, free = shutil.disk_usage(BASE)
print(f'\n✓ Environnement prêt ! Espace libre : {free / 1024**3:.1f} Go')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 3 — VÉRIFICATION DE L'ENVIRONNEMENT PYTHON 3.10
# ══════════════════════════════════════════════════
import os, subprocess

BASE  = open('/tmp/abiart_base.txt').read().strip()
WEBUI = f'{BASE}/stable-diffusion-webui'
PY    = f'{BASE}/venv/bin/python'

# Environnement propre : pas de PYTHONPATH Kaggle, pas de bibliothèques CUDA système
# (elles entreraient en conflit avec celles embarquées dans torch 2.1.2)
env = os.environ.copy()
for k in ('PYTHONPATH', 'PYTHONHOME', 'VIRTUAL_ENV'):
    env.pop(k, None)
env['LD_LIBRARY_PATH'] = ':'.join(p for p in env.get('LD_LIBRARY_PATH', '').split(':') if p and 'cuda' not in p.lower())
env['CUDA_VISIBLE_DEVICES'] = '0'
env['MPLBACKEND'] = 'Agg'  # Kaggle force un backend matplotlib Jupyter absent du venv 3.10

test = r"""
import sys, importlib
print('Python        :', sys.version.split()[0])
import torch
print('torch         :', torch.__version__, '| CUDA', torch.version.cuda)
print('GPU dispo     :', torch.cuda.is_available())
assert torch.cuda.is_available(), "torch ne voit pas le GPU"
print('GPU           :', torch.cuda.get_device_name(0))
x = torch.randn(1024, 1024, device='cuda', dtype=torch.float16)
print('Test calcul   :', 'OK' if torch.isfinite((x @ x).sum()).item() else 'NaN')
bad = 0
for m in ['transformers', 'gradio', 'fastapi', 'pytorch_lightning', 'open_clip', 'clip', 'safetensors',
          'accelerate', 'omegaconf', 'kornia', 'einops', 'torchsde', 'spandrel', 'diskcache', 'skimage']:
    try:
        mod = importlib.import_module(m)
        print(f'  ✓ {m:<18}', getattr(mod, '__version__', ''))
    except Exception as e:
        bad += 1
        print(f'  ❌ {m:<18}', e)
for m in ['ultralytics', 'cv2']:   # ADetailer : facultatif, ne bloque pas
    try:
        mod = importlib.import_module(m)
        print(f'  ✓ {m:<18}', getattr(mod, '__version__', ''), '(ADetailer)')
    except Exception as e:
        print(f'  ⚠️ {m:<18}', e, '-> ADetailer indisponible (le reste fonctionne)')
sys.exit(1 if bad else 0)
"""
r = subprocess.run([PY, '-c', test], env=env, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError('Environnement incomplet : relancez la cellule 2, puis celle-ci.')
print('✓ Environnement Python 3.10 opérationnel.')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 4 — MODÈLES (Animagine XL 3.1 + SDXL VAE fp16-fix + ADetailer + restauration Hugging Face)
# ══════════════════════════════════════════════════
import os, shutil, subprocess

BASE  = open('/tmp/abiart_base.txt').read().strip()
WEBUI = f'{BASE}/stable-diffusion-webui'
ckpt_dir = f'{WEBUI}/models/Stable-diffusion'
vae_dir  = f'{WEBUI}/models/VAE'
os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(vae_dir, exist_ok=True)

def download(url, dest, min_bytes, label):
    if os.path.exists(dest) and os.path.getsize(dest) >= min_bytes:
        print(f'✓ {label} déjà présent')
        return
    print(f'Téléchargement de {label}...')
    for attempt in range(1, 4):
        r = subprocess.run(['wget', '-c', '-q', '--show-progress', '--progress=dot:giga',
                            '--tries=3', '--timeout=60', url, '-O', dest])
        if r.returncode == 0 and os.path.getsize(dest) >= min_bytes:
            print(f'✓ {label} OK ({os.path.getsize(dest) / 1024**3:.2f} Go)')
            return
        print(f'  ⚠️ tentative {attempt}/3 échouée, nouvel essai...')
    raise RuntimeError(f'Téléchargement de {label} impossible (Internet est-il sur ON ?)')

download('https://huggingface.co/cagliostrolab/animagine-xl-3.1/resolve/main/animagine-xl-3.1.safetensors',
         f'{ckpt_dir}/animagine-xl-3.1.safetensors', 6_000_000_000, 'Animagine XL 3.1 (~6,9 Go)')

# VAE "fp16-fix" : le VAE SDXL d'origine produit des images noires/NaN en demi-précision sur T4/P100
download('https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl.vae.safetensors',
         f'{vae_dir}/sdxl_vae_fp16_fix.safetensors', 300_000_000, 'SDXL VAE fp16-fix (~335 Mo)')

# Modèles ADetailer supplémentaires (face_yolov9c / hand_yolov9c, proposés dans l'onglet Kaggle).
# ADetailer ne télécharge automatiquement que les yolov8 : on ajoute ici les yolov9c.
# Facultatif : en cas d'échec, les modèles yolov8 restent utilisables.
ad_dir = f'{WEBUI}/models/adetailer'
os.makedirs(ad_dir, exist_ok=True)
for name in ('face_yolov9c.pt', 'hand_yolov9c.pt'):
    try:
        download(f'https://huggingface.co/Bingsu/adetailer/resolve/main/{name}',
                 f'{ad_dir}/{name}', 10_000_000, f'ADetailer {name}')
    except Exception as e:
        print(f'  ⚠️ {name} non téléchargé ({e}) : ce modèle sera indisponible dans ADetailer.')

# Upscalers pour le Hi-Res Fix (ils apparaissent dans la liste "Upscaler" du site).
# "R-ESRGAN 4x+" est pensé pour la photo et durcit les traits ; ceux-ci conviennent mieux à l'illustration.
# Facultatif : en cas d'échec, les upscalers intégrés d'A1111 restent utilisables.
esrgan_dir = f'{WEBUI}/models/ESRGAN'
os.makedirs(esrgan_dir, exist_ok=True)
for up_name, up_url in (
        ('4x-AnimeSharp.pth', 'https://huggingface.co/Kim2091/AnimeSharp/resolve/main/4x-AnimeSharp.pth'),
        ('4x-UltraSharp.pth', 'https://huggingface.co/lokCX/4x-Ultrasharp/resolve/main/4x-UltraSharp.pth'),
        ('4x_foolhardy_Remacri.pth', 'https://huggingface.co/FacehugmanIII/4x_foolhardy_Remacri/resolve/main/4x_foolhardy_Remacri.pth')):
    try:
        download(up_url, f'{esrgan_dir}/{up_name}', 30_000_000, f'Upscaler {up_name}')
    except Exception as e:
        print(f'  ⚠️ {up_name} non téléchargé ({e}) : cet upscaler sera absent de la liste Hi-Res.')

# ── Restauration des modèles sauvegardés sur Hugging Face (dépôt privé) ──
# Secrets Kaggle (Add-ons → Secrets) : HF_TOKEN (jeton avec droit d'ÉCRITURE) et HF_REPO (ex. monpseudo/abiart-models).
# Le dépôt contient les dossiers checkpoints/, loras/ et vae/ ; il se remplit automatiquement à chaque
# téléchargement fait depuis le site AbiArt (voir cellule 5).
def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return (UserSecretsClient().get_secret(name) or '').strip()
    except Exception:
        return os.environ.get(name, '').strip()

HF_TOKEN = get_secret('HF_TOKEN')
HF_REPO  = get_secret('HF_REPO')
if HF_TOKEN and HF_REPO:
    print(f'\nRestauration des modèles depuis Hugging Face ({HF_REPO})...')
    HF_FOLDERS = {'checkpoints': 'Stable-diffusion', 'loras': 'Lora', 'vae': 'VAE', 'embeddings': '../embeddings'}
    try:
        from huggingface_hub import HfApi, hf_hub_download
        try:
            files = HfApi(token=HF_TOKEN).list_repo_files(HF_REPO, repo_type='model')
        except Exception as e:
            files = []
            if type(e).__name__ == 'RepositoryNotFoundError':
                print('  (dépôt introuvable : il sera créé au premier modèle sauvegardé si le jeton est valide)')
            else:
                raise
        todo = []
        for f in files:
            folder, _, fname = f.partition('/')
            if folder in HF_FOLDERS and fname and '/' not in fname \
                    and fname.lower().endswith(('.safetensors', '.ckpt', '.pt', '.pth')):
                todo.append((f, f'{WEBUI}/models/{HF_FOLDERS[folder]}/{fname}'))
        tmp_dir = f'{BASE}/hf_restore'
        for f, dest in todo:
            label = os.path.basename(dest)
            if os.path.exists(dest) and os.path.getsize(dest) > 0:
                print(f'  ✓ {label} déjà présent')
                continue
            try:
                print(f'  ↓ {label}...')
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                src = hf_hub_download(repo_id=HF_REPO, filename=f, token=HF_TOKEN, local_dir=tmp_dir)
                shutil.move(src, dest)
                print(f'  ✓ {label} restauré')
            except Exception as e:
                print(f'  ⚠️ {label} non restauré : {str(e)[:200]}')
        shutil.rmtree(tmp_dir, ignore_errors=True)
    except Exception as e:
        print(f'  ⚠️ Restauration Hugging Face impossible : {str(e)[:300]}')
else:
    print('\nℹ️ Sauvegarde Hugging Face non configurée (secrets HF_TOKEN et HF_REPO absents) : '
          'les modèles ajoutés seront perdus au redémarrage.')

total, used, free = shutil.disk_usage(BASE)
print(f'\n✓ Modèles prêts ! Espace libre : {free / 1024**3:.1f} Go')

In [ ]:
# ══════════════════════════════════════════════════
# CELLULE 5 — LANCEMENT DE L'API A1111 + TUNNEL CLOUDFLARE
# (le tunnel n'est ouvert qu'une fois le modèle réellement chargé)
# ══════════════════════════════════════════════════
import os, re, sys, time, shutil, signal, subprocess, requests

BASE  = open('/tmp/abiart_base.txt').read().strip()
WEBUI = f'{BASE}/stable-diffusion-webui'
PY    = f'{BASE}/venv/bin/python'
PORT  = 7860
LOG   = '/tmp/webui.log'
TLOG  = '/tmp/tunnel.log'
CF    = '/tmp/cloudflared'

# Optionnel : URL de votre site AbiArt déployé, pour l'auto-connexion
# Ex: ABIART_APP_URL = "https://mon-vrai-site.up.railway.app"
ABIART_APP_URL = ""

ckpt = f'{WEBUI}/models/Stable-diffusion/animagine-xl-3.1.safetensors'
vae  = f'{WEBUI}/models/VAE/sdxl_vae_fp16_fix.safetensors'
if not os.path.exists(ckpt):
    raise RuntimeError('Modèle absent : exécutez la cellule 4.')

# 0. Nettoyage d'une éventuelle exécution précédente
subprocess.run(['pkill', '-9', '-f', f'{WEBUI}/launch.py'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['pkill', '-9', '-f', 'launch.py --nowebui'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['pkill', '-9', '-f', 'cloudflared'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

# 1. Environnement propre (voir cellule 3)
env = os.environ.copy()
for k in ('PYTHONPATH', 'PYTHONHOME', 'VIRTUAL_ENV'):
    env.pop(k, None)
env['LD_LIBRARY_PATH'] = ':'.join(p for p in env.get('LD_LIBRARY_PATH', '').split(':') if p and 'cuda' not in p.lower())
env['PATH'] = f'{BASE}/venv/bin:' + env.get('PATH', '')
env['CUDA_VISIBLE_DEVICES'] = '0'
env['MPLBACKEND'] = 'Agg'  # Kaggle force un backend matplotlib Jupyter absent du venv 3.10
env['PYTHONUNBUFFERED'] = '1'
env['STABLE_DIFFUSION_REPO'] = 'https://github.com/w-e-w/stablediffusion.git'
env['ABIART_LOCAL_PORT'] = str(PORT)  # lu par l'extension abiart-downloader (generation asynchrone)

# 1c. Second GPU (Kaggle "T4 x2") : une 2e instance A1111 tourne sur le GPU 1 (port PORT+1).
#     L'extension répartit les générations entre les deux : 2 images peuvent être calculées en même temps.
#     Mettre USE_GPU2 = False pour revenir au fonctionnement à un seul GPU.
try:
    _g = subprocess.run(['nvidia-smi', '--query-gpu=index', '--format=csv,noheader'], capture_output=True, text=True)
    N_GPU = len([l for l in _g.stdout.splitlines() if l.strip()]) if _g.returncode == 0 else 1
except Exception:
    N_GPU = 1
USE_GPU2 = N_GPU >= 2
PORT2 = PORT + 1
LOG2 = '/tmp/webui_gpu1.log'
env['ABIART_BACKEND_PORTS'] = f'{PORT},{PORT2}' if USE_GPU2 else str(PORT)
print(f'🖥️  GPU détectés : {N_GPU} → ' + ('2 instances A1111 (une par GPU)' if USE_GPU2 else '1 instance A1111'))

# 1a. Sauvegarde Hugging Face : l'extension y envoie chaque nouveau modèle téléchargé depuis le site
def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return (UserSecretsClient().get_secret(name) or '').strip()
    except Exception:
        return os.environ.get(name, '').strip()

_hf_token, _hf_repo = _secret('HF_TOKEN'), _secret('HF_REPO')
if _hf_token and _hf_repo:
    env['ABIART_HF_TOKEN'] = _hf_token
    env['ABIART_HF_REPO'] = _hf_repo
    print(f'💾 Sauvegarde Hugging Face activée → {_hf_repo}')
else:
    print('ℹ️ Sauvegarde Hugging Face désactivée (secrets HF_TOKEN / HF_REPO absents).')

# 1b. Extension AbiArt : permet au site de télécharger des modèles Civitai / Hugging Face sur cette machine
EXT_DIR = f'{WEBUI}/extensions/abiart-downloader/scripts'
os.makedirs(EXT_DIR, exist_ok=True)
EXT_SRC = r"""# AbiArt - telechargement de modeles (Civitai / Hugging Face) vers A1111 depuis le site.
#
# Ajoute 3 routes a l'API d'Automatic1111 (meme port, donc meme tunnel Cloudflare) :
#   GET  /abiart/ping                 -> detecte la presence de l'extension
#   POST /abiart/download             -> demarre un telechargement en arriere-plan, renvoie un job_id
#   GET  /abiart/download/{job_id}    -> progression (running / done / error)
#
# Securite : https uniquement, hotes de depart limites (civitai.com, civitai.red, huggingface.co),
# redirections suivies a la main (adresses privees refusees), extensions de fichier et taille limitees.
import ipaddress
import os
import re
import shutil
import socket
import threading
import time
import uuid
from urllib.parse import urljoin, urlparse

import requests

ALLOWED_HOSTS = ("civitai.com", "civitai.red", "huggingface.co")
ALLOWED_EXT = (".safetensors", ".ckpt", ".pt", ".pth")
MAX_BYTES = 14 * 1024 ** 3
MAX_ACTIVE_JOBS = 2
SUBDIRS = {"checkpoint": "Stable-diffusion", "lora": "Lora", "vae": "VAE", "embedding": "../embeddings"}
HF_TOKEN = os.environ.get("ABIART_HF_TOKEN", "")
HF_REPO = os.environ.get("ABIART_HF_REPO", "")
HF_FOLDERS = {"checkpoint": "checkpoints", "lora": "loras", "vae": "vae", "embedding": "embeddings"}

JOBS = {}
_LOCK = threading.Lock()


def _models_path():
    try:
        from modules import paths_internal
        return paths_internal.models_path
    except Exception:
        return os.environ.get("ABIART_MODELS_PATH", "models")


def _host_allowed(host):
    host = (host or "").lower()
    return any(host == d or host.endswith("." + d) for d in ALLOWED_HOSTS)


def _is_public(host):
    try:
        infos = socket.getaddrinfo(host, 443, proto=socket.IPPROTO_TCP)
    except Exception:
        return False
    return bool(infos) and all(ipaddress.ip_address(i[4][0]).is_global for i in infos)


def _safe_filename(name):
    name = os.path.basename((name or "").replace("\\", "/")).strip()
    name = re.sub(r"[^A-Za-z0-9._\-\(\) \[\]]", "_", name).lstrip(".")[:150]
    if not name.lower().endswith(ALLOWED_EXT):
        raise ValueError("Unsupported file type (allowed: " + ", ".join(ALLOWED_EXT) + ")")
    return name


def _clean_name(name):
    # Comme _safe_filename, mais renvoie "" (au lieu de lever) si le nom n'est pas un fichier de modele.
    try:
        return _safe_filename(name)
    except ValueError:
        return ""


def _open(url, token=""):
    # GET en streaming ; les redirections sont suivies a la main pour verifier chaque saut.
    current = url
    for hop in range(6):
        p = urlparse(current)
        host = p.hostname or ""
        if p.scheme != "https":
            raise ValueError("Only https URLs are allowed")
        if hop == 0 and not _host_allowed(host):
            raise ValueError("Host not allowed: " + host)
        if not _is_public(host):
            raise ValueError("Cannot reach %s from this machine, or its address is not public" % host)
        headers = {"User-Agent": "AbiArt-Kaggle/1.0"}
        if hop == 0 and token and "civitai" in host:
            headers["Authorization"] = "Bearer " + token  # jamais transmis apres une redirection
        r = requests.get(current, headers=headers, stream=True, allow_redirects=False, timeout=(15, 60))
        if r.status_code in (301, 302, 303, 307, 308) and r.headers.get("Location"):
            current = urljoin(current, r.headers["Location"])
            r.close()
            continue
        return r
    raise ValueError("Too many redirects")


def _header_filename(resp):
    cd = resp.headers.get("Content-Disposition", "")
    m = re.search(r"filename\*?=(?:UTF-8'')?\"?([^\";]+)\"?", cd, re.I)
    return m.group(1) if m else ""


def _backup_to_hf(job, final, kind, name):
    # Sauvegarde le modele dans le depot Hugging Face prive (si configure), pour le retrouver apres un redemarrage.
    if not (HF_TOKEN and HF_REPO):
        job["backup"] = "disabled"
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
        job["backup"] = "running"
        print("[AbiArt] Sauvegarde Hugging Face en cours : %s" % name)
        api.upload_file(path_or_fileobj=final, path_in_repo="%s/%s" % (HF_FOLDERS[kind], name),
                        repo_id=HF_REPO, repo_type="model", commit_message="Add " + name)
        job["backup"] = "done"
        print("[AbiArt] Sauvegarde Hugging Face terminee : %s" % name)
    except Exception as e:
        job["backup"] = "error"
        print("[AbiArt] Sauvegarde Hugging Face impossible pour %s : %s" % (name, str(e)[:300]))


def _worker(job_id, url, token, kind, wanted_name):
    job = JOBS[job_id]
    tmp = None
    try:
        r = _open(url, token)
        if r.status_code in (401, 403) and token and "token=" not in url:
            # Civitai accepte le token en en-tete (Bearer) ou en parametre d'URL : on essaie l'autre methode.
            r.close()
            print("[AbiArt] Civitai a refuse l'en-tete Authorization (HTTP %d) : nouvel essai avec ?token=" % r.status_code)
            r = _open(url + ("&" if "?" in url else "?") + "token=" + token, token)
        if r.status_code in (401, 403):
            try:
                detail = " ".join((r.text or "")[:200].split())
            except Exception:
                detail = ""
            where = urlparse(getattr(r, "url", "") or url).hostname or ""
            print("[AbiArt] Refus HTTP %d de %s : %s" % (r.status_code, where, detail))
            if token:
                raise RuntimeError("Download refused (HTTP %d) : le notebook a recu un token Civitai se terminant par "
                                   "...%s et Civitai le refuse (reponse de %s : %s). Compare ces 4 derniers caracteres "
                                   "avec ta cle : s'ils sont differents, le site envoie un autre token que celui du champ "
                                   "(variable CIVITAI_TOKEN de Railway ?)."
                                   % (r.status_code, token[-4:], where, detail or "vide"))
            raise RuntimeError("Download refused (HTTP %d) : aucun token Civitai recu par le notebook. Colle-le dans le "
                               "champ \"Civitai API Token\" du site, puis relance (et verifie que app.py / index.html "
                               "du site sont a jour)." % r.status_code)
        if r.status_code != 200:
            raise RuntimeError("Download failed (HTTP %d)" % r.status_code)
        if "text/html" in r.headers.get("Content-Type", ""):
            raise RuntimeError("The link returned a web page, not a model file (login required?)")
        header_name = _header_filename(r)
        name = _clean_name(wanted_name) or _clean_name(header_name)
        if not name:
            if header_name:  # Civitai envoie un fichier qui n'est pas un modele (zip, yaml, ...)
                raise RuntimeError('Ce lien ne pointe pas vers un modele : fichier recu "%s" (attendu : %s)'
                                   % (header_name, ", ".join(ALLOWED_EXT)))
            name = "model_%s.safetensors" % job_id[:6]
        total = int(r.headers.get("Content-Length") or 0)
        if total > MAX_BYTES:
            raise RuntimeError("File too large")
        dest_dir = os.path.normpath(os.path.join(_models_path(), SUBDIRS[kind]))
        os.makedirs(dest_dir, exist_ok=True)
        if total and shutil.disk_usage(dest_dir).free < total + 1024 ** 3:
            raise RuntimeError("Not enough free disk space on the Kaggle machine")
        final = os.path.join(dest_dir, name)
        job.update(filename=name, total=total)
        if os.path.exists(final) and os.path.getsize(final) > 0:
            size = os.path.getsize(final)
            job.update(state="done", done=size, total=size, note="already present")
            return
        tmp = final + ".part"
        done = 0
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                if not chunk:
                    continue
                f.write(chunk)
                done += len(chunk)
                if done > MAX_BYTES:
                    raise RuntimeError("File too large")
                job["done"] = done
        if total and done != total:
            raise RuntimeError("Incomplete download")
        os.replace(tmp, final)
        tmp = None
        job.update(state="done", done=done, total=done)
        _backup_to_hf(job, final, kind, name)
    except Exception as e:
        job.update(state="error", error=str(e))
    finally:
        if tmp and os.path.exists(tmp):
            try:
                os.remove(tmp)
            except OSError:
                pass


def start_download(data):
    # Retourne (payload, http_status).
    url = (data.get("url") or "").strip()
    kind = (data.get("type") or "checkpoint").lower()
    if kind not in SUBDIRS:
        return {"error": "Unknown type: " + kind}, 400
    if not url:
        return {"error": "Missing url"}, 400
    # Nom suggere par le site : garde s'il est valide, sinon ignore (le vrai nom est lu dans la reponse de Civitai).
    wanted = _clean_name((data.get("filename") or "").strip())
    host = urlparse(url).hostname or ""
    if urlparse(url).scheme != "https" or not _host_allowed(host):
        return {"error": "Only https links from civitai.com, civitai.red or huggingface.co are allowed"}, 400
    with _LOCK:
        if sum(1 for j in JOBS.values() if j["state"] == "running") >= MAX_ACTIVE_JOBS:
            return {"error": "Too many downloads in progress, wait for one to finish"}, 429
        job_id = uuid.uuid4().hex[:12]
        JOBS[job_id] = {"state": "running", "kind": kind, "filename": wanted, "done": 0, "total": 0,
                        "error": "", "started": time.time()}
    token = (data.get("token") or "").strip().strip("\"'")
    if token.lower().startswith("bearer "):
        token = token[7:].strip()
    print("[AbiArt] Telechargement %s (%s) - token Civitai recu : %s" % (wanted or url[:60], kind, ("oui, se termine par ..." + token[-4:]) if token else "NON"))
    threading.Thread(target=_worker, args=(job_id, url, token, kind, wanted),
                     daemon=True).start()
    return {"success": True, "job_id": job_id, "filename": wanted, "type": kind}, 200


def job_status(job_id):
    job = JOBS.get(job_id)
    if not job:
        return {"error": "Unknown job (the notebook may have been restarted)", "state": "error"}, 404
    return {k: job.get(k) for k in ("state", "kind", "filename", "done", "total", "error", "note", "backup")}, 200


# ---------------------------------------------------------------------------------------------
# Generation asynchrone : evite la coupure a ~100 s du tunnel Cloudflare (erreur HTTP 524).
# Le site demarre la generation (reponse immediate : un job_id) puis demande l'etat toutes les 2 s.
# Aucune requete HTTP ne dure plus de quelques secondes, quelle que soit la duree du calcul.
# ---------------------------------------------------------------------------------------------
# Une instance A1111 par GPU : BACKENDS[0] est cette instance (celle du tunnel), les suivantes sont
# joignables en local. Chaque instance calcule une image a la fois ; les jobs sont servis dans l'ordre d'arrivee.
_PORTS = [p.strip() for p in os.environ.get("ABIART_BACKEND_PORTS", os.environ.get("ABIART_LOCAL_PORT", "7860")).split(",")
          if p.strip()]
BACKENDS = [{"id": i, "url": "http://127.0.0.1:" + p, "busy": False, "up": i == 0, "checked": 0.0}
            for i, p in enumerate(_PORTS)]
LOCAL_API = BACKENDS[0]["url"]
GEN_TTL = 15 * 60
GEN_MAX_ACTIVE = 2 + 2 * len(BACKENDS)
GEN_JOBS = {}
_GEN_LOCK = threading.Lock()
_BK_COND = threading.Condition()
_LOAD_LOCK = threading.Lock()
_WAITING = []   # job_id en attente d'un GPU, dans l'ordre d'arrivee


def _gen_error(resp):
    try:
        js = resp.json()
        msg = js.get("error") or js.get("detail") or js.get("errors") or str(js)
    except Exception:
        msg = resp.text or ("HTTP %d" % resp.status_code)
    return str(msg)[:400]


def _backend_up(b):
    # L'instance 0 est ce processus. Les autres sont testees au plus toutes les 10 s tant qu'elles ne repondent pas
    # (elles demarrent apres l'instance 0, et peuvent ne jamais demarrer).
    if b["id"] == 0 or b["up"]:
        return True
    now = time.time()
    if now - b["checked"] < 10:
        return False
    b["checked"] = now
    try:
        b["up"] = requests.get(b["url"] + "/sdapi/v1/options", timeout=3).status_code == 200
    except Exception:
        b["up"] = False
    return b["up"]


def _current_model(b):
    # Checkpoint charge sur une instance ("" si inconnu, None si l'instance ne repond pas).
    try:
        return requests.get(b["url"] + "/sdapi/v1/options", timeout=10).json().get("sd_model_checkpoint") or ""
    except Exception:
        return None


def _acquire(job_id, wanted):
    # Attend un GPU libre. A choix egal, prend celui qui a deja le bon checkpoint charge (pas de rechargement).
    job = GEN_JOBS[job_id]
    with _BK_COND:
        _WAITING.append(job_id)
        try:
            while True:
                if job.get("cancel"):
                    return None
                if _WAITING[0] == job_id:
                    free = [b for b in BACKENDS if not b["busy"] and _backend_up(b)]
                    if free:
                        pick = free[0]
                        if wanted and len(free) > 1:
                            for b in free:
                                if _current_model(b) == wanted:
                                    pick = b
                                    break
                        pick["busy"] = True
                        return pick
                _BK_COND.wait(timeout=1.0)
        finally:
            _WAITING.remove(job_id)
            _BK_COND.notify_all()


def _release(b):
    with _BK_COND:
        b["busy"] = False
        _BK_COND.notify_all()


def _run_on(job, b, payload, wanted):
    # Calcule le job sur l'instance b. Renvoie True s'il faut le relancer ailleurs (instance secondaire tombee).
    job.update(state="running", backend=b["id"], started=time.time(), updated=time.time())
    try:
        target = wanted
        if not target and b["id"] != 0:
            # Ancien site (il change le checkpoint de l'instance 0 lui-meme) : l'instance secondaire la suit.
            target = _current_model(BACKENDS[0]) or ""
        if target:
            current = _current_model(b)
            if current is None:
                raise requests.exceptions.ConnectionError("backend %d unreachable" % b["id"])
            if current != target:
                job["loading_model"] = True
                # Un seul chargement de checkpoint a la fois sur la machine : chaque chargement prend
                # plusieurs Go de memoire vive le temps de lire le fichier.
                with _LOAD_LOCK:
                    r = requests.post(b["url"] + "/sdapi/v1/options", json={"sd_model_checkpoint": target}, timeout=900)
                job["loading_model"] = False
                if r.status_code != 200:
                    job.update(state="error", error="Model switch failed: " + _gen_error(r), updated=time.time())
                    return False
        if job.get("cancel"):
            job.update(state="cancelled", error="Cancelled", updated=time.time())
            return False
        t_gen = time.time()
        job["load_seconds"] = round(t_gen - job["started"], 1)   # changement de checkpoint eventuel
        r = requests.post(b["url"] + "/sdapi/v1/txt2img", json=payload, timeout=3600)
        if job.get("cancel"):
            job.update(state="cancelled", error="Cancelled", updated=time.time())
        elif r.status_code == 200:
            data = r.json()
            job.update(state="done", images=data.get("images", []), info=data.get("info"), updated=time.time(),
                       seconds=round(time.time() - t_gen, 1))   # calcul seul, hors file d'attente
        else:
            job.update(state="error", error=_gen_error(r), updated=time.time())
        return False
    except requests.exceptions.ConnectionError as e:
        job["loading_model"] = False
        if b["id"] != 0 and not job.get("cancel"):
            b["up"] = False
            b["checked"] = time.time()
            print("[AbiArt] GPU %d ne repond plus : le job est relance sur un autre GPU" % b["id"])
            job.update(state="queued", updated=time.time())
            return True
        job.update(state="error", error=("%s: %s" % (type(e).__name__, e))[:400], updated=time.time())
        return False
    except Exception as e:
        job["loading_model"] = False
        job.update(state="error", error=("%s: %s" % (type(e).__name__, e))[:400], updated=time.time())
        return False


def _gen_worker(job_id, payload):
    job = GEN_JOBS[job_id]
    # "abiart_model" : checkpoint demande par le site pour CE job (absent avec un ancien site).
    wanted = str(payload.pop("abiart_model", "") or "").strip()
    for _attempt in range(len(BACKENDS) + 1):
        b = _acquire(job_id, wanted)
        if b is None:
            job.update(state="cancelled", error="Cancelled", updated=time.time())
            return
        try:
            retry = _run_on(job, b, payload, wanted)
        finally:
            _release(b)
        if not retry:
            return
    job.update(state="error", error="No GPU backend available", updated=time.time())


def start_generate(data):
    # Retourne (payload, http_status). `data` = corps d'un appel /sdapi/v1/txt2img (+ "abiart_model" facultatif).
    if not isinstance(data, dict) or not data.get("prompt"):
        return {"error": "Missing prompt"}, 400
    now = time.time()
    with _GEN_LOCK:
        for jid in [j for j, v in GEN_JOBS.items() if v["state"] in ("done", "error", "cancelled") and now - v["updated"] > GEN_TTL]:
            GEN_JOBS.pop(jid, None)
        if sum(1 for v in GEN_JOBS.values() if v["state"] in ("queued", "running")) >= GEN_MAX_ACTIVE:
            return {"error": "Too many generations in progress, wait for one to finish"}, 429
        job_id = uuid.uuid4().hex[:16]
        GEN_JOBS[job_id] = {"state": "queued", "created": now, "updated": now, "error": "", "cancel": False}
    threading.Thread(target=_gen_worker, args=(job_id, dict(data)), daemon=True).start()
    return {"success": True, "job_id": job_id}, 200


def gen_status(job_id):
    job = GEN_JOBS.get(job_id)
    if not job:
        return {"error": "Unknown job (the notebook may have been restarted)", "state": "error"}, 404
    out = {"state": job["state"], "error": job.get("error", "")}
    if job["state"] == "queued":
        out["queue_position"] = sum(1 for v in GEN_JOBS.values()
                                    if v["state"] == "queued" and v["created"] < job["created"])
    elif job["state"] == "running":
        out["gpu"] = job.get("backend", 0)
        if job.get("loading_model"):
            out["progress"] = 0.0
            out["loading_model"] = True
        else:
            try:
                api = BACKENDS[job.get("backend", 0)]["url"]
                p = requests.get(api + "/sdapi/v1/progress", params={"skip_current_image": "true"}, timeout=5).json()
                out["progress"] = float(p.get("progress") or 0)
                out["eta"] = p.get("eta_relative")
            except Exception:
                out["progress"] = None
    elif job["state"] == "done":
        out["progress"] = 1.0
        out["gpu"] = job.get("backend", 0)
        out["seconds"] = job.get("seconds")
        out["load_seconds"] = job.get("load_seconds")
        out["images"] = job["images"]
        out["info"] = job["info"]
    return out, 200


def gen_cancel(job_id):
    job = GEN_JOBS.get(job_id)
    if not job:
        return {"error": "Unknown job"}, 404
    if job["state"] in ("queued", "running"):
        job["cancel"] = True
        if job["state"] == "queued":
            job.update(state="cancelled", error="Cancelled", updated=time.time())
        else:
            try:
                requests.post(BACKENDS[job.get("backend", 0)]["url"] + "/sdapi/v1/interrupt", timeout=10)
            except Exception:
                pass
    return {"success": True}, 200


def _register(_demo, app):
    from fastapi import Request
    from fastapi.responses import JSONResponse

    @app.get("/abiart/ping")
    async def abiart_ping():
        return {"ok": True, "version": 3, "jobs": True, "routing": True,
                "gpus": sum(1 for b in BACKENDS if _backend_up(b))}

    @app.post("/abiart/download")
    async def abiart_download(request: Request):
        try:
            data = await request.json()
        except Exception:
            data = {}
        payload, status = start_download(data if isinstance(data, dict) else {})
        return JSONResponse(payload, status_code=status)

    @app.get("/abiart/download/{job_id}")
    async def abiart_download_status(job_id: str):
        payload, status = job_status(job_id)
        return JSONResponse(payload, status_code=status)

    import asyncio

    async def _in_thread(fn, *args):
        return await asyncio.get_running_loop().run_in_executor(None, fn, *args)

    @app.post("/abiart/generate")
    async def abiart_generate(request: Request):
        try:
            data = await request.json()
        except Exception:
            data = {}
        payload, status = await _in_thread(start_generate, data if isinstance(data, dict) else {})
        return JSONResponse(payload, status_code=status)

    @app.get("/abiart/generate/{job_id}")
    async def abiart_generate_status(job_id: str):
        payload, status = await _in_thread(gen_status, job_id)
        return JSONResponse(payload, status_code=status)

    @app.post("/abiart/generate/{job_id}/cancel")
    async def abiart_generate_cancel(job_id: str):
        payload, status = await _in_thread(gen_cancel, job_id)
        return JSONResponse(payload, status_code=status)


try:
    from modules import script_callbacks
    script_callbacks.on_app_started(_register)
except Exception:
    pass  # import hors A1111 (tests)
"""
with open(f'{EXT_DIR}/abiart_downloader.py', 'w', encoding='utf-8') as f:
    f.write(EXT_SRC)

# 2. Lancement d'A1111 en mode API seule (processus d'arrière-plan, logs dans un fichier)
# --medvram-sdxl garde une copie du modèle en mémoire vive (~7 Go par instance) : à deux instances, la RAM
# de Kaggle déborde et la session redémarre. Sans cette option le modèle reste sur la carte graphique
# (16 Go sur un T4) et les générations sont un peu plus rapides. Mettre MEDVRAM = True seulement en cas
# d'erreur "CUDA out of memory" en Hi-Res : le second GPU est alors désactivé automatiquement.
MEDVRAM = False
if MEDVRAM and USE_GPU2:
    USE_GPU2 = False
    env['ABIART_BACKEND_PORTS'] = str(PORT)
    print('ℹ️ MEDVRAM = True : second GPU désactivé (pas assez de mémoire vive pour deux instances).')

def ram_free_gb():
    try:
        for line in open('/proc/meminfo'):
            if line.startswith('MemAvailable:'):
                return int(line.split()[1]) / 1024 ** 2
    except Exception:
        pass
    return 0.0

def launch_webui(port, gpu, log_path, extra=()):
    cmd = [PY, 'launch.py',
           '--nowebui', '--api', '--port', str(port),
           '--skip-install',
           '--opt-sdp-attention', '--no-hashing',
           '--cors-allow-origins=*',
           '--ckpt', ckpt]
    if MEDVRAM:
        cmd.append('--medvram-sdxl')
    if os.path.exists(vae):
        cmd += ['--vae-path', vae]
    cmd += list(extra)
    e = dict(env)
    e['CUDA_VISIBLE_DEVICES'] = str(gpu)
    return subprocess.Popen(cmd, cwd=WEBUI, env=e, stdout=open(log_path, 'w'), stderr=subprocess.STDOUT,
                            start_new_session=True)

proc = launch_webui(PORT, 0, LOG)
print('Démarrage d\'Automatic1111 (chargement du modèle : 1 à 3 min)...\n')

def start_tunnel():
    if not os.path.exists(CF):
        print('Téléchargement de cloudflared...')
        subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
                        '-O', CF], check=True)
        os.chmod(CF, 0o755)
    tl = open(TLOG, 'w')
    tp = subprocess.Popen([CF, 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{PORT}'],
                          stdout=tl, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        time.sleep(1)
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', open(TLOG).read())
        if m:
            return tp, m.group(0)
    return tp, None

pos, ready, tunnel_proc, url, t0 = 0, False, None, None, time.time()
proc2, ready2, t2 = None, False, 0.0
try:
    while True:
        # a) afficher les nouvelles lignes du log
        with open(LOG, 'r', errors='replace') as f:
            f.seek(pos)
            chunk = f.read()
            pos = f.tell()
        if chunk:
            print(chunk, end='')

        # b) le serveur est-il mort ?
        if proc.poll() is not None:
            print(f'\n❌ Le serveur A1111 s\'est arrêté (code {proc.returncode}). Voir les logs ci-dessus.')
            break

        if not ready:
            full = open(LOG, errors='replace').read()
            if 'stable diffusion model failed to load' in full.lower():
                print('\n❌ Le modèle Stable Diffusion n\'a pas pu se charger. Voir les logs ci-dessus.')
                proc.terminate()
                break
            if 'Model loaded in' in full:
                try:
                    ready = requests.get(f'http://127.0.0.1:{PORT}/sdapi/v1/options', timeout=5).status_code == 200
                except Exception:
                    ready = False
            if not ready and time.time() - t0 > 1500:
                print('\n❌ Délai dépassé (25 min) : le serveur n\'est jamais devenu prêt.')
                proc.terminate()
                break

            if ready:
                # c) ouvrir le tunnel Cloudflare maintenant que l'API répond
                tunnel_proc, url = start_tunnel()
                if url:
                    # laisser le DNS se propager puis tester l'URL publique
                    ok = False
                    for _ in range(15):
                        try:
                            if requests.get(url + '/sdapi/v1/sd-models', timeout=10).status_code == 200:
                                ok = True
                                break
                        except Exception:
                            pass
                        time.sleep(2)
                    print('\n' + '=' * 65)
                    print('🎨 BACKEND KAGGLE PRÊT :')
                    print(f'   >>>  {url}  <<<')
                    print('=' * 65)
                    if not ok:
                        print('⚠️ L\'URL ne répond pas encore au test, réessayez dans quelques secondes.')
                    try:
                        _ping = requests.get(f'http://127.0.0.1:{PORT}/abiart/ping', timeout=5)
                        ext_ok = _ping.status_code == 200
                        jobs_ok = ext_ok and bool(_ping.json().get('jobs'))
                    except Exception:
                        ext_ok = False
                        jobs_ok = False
                    print('\n📥 Téléchargement de modèles Civitai depuis le site : ' + ('activé ✓' if ext_ok else '⚠️ extension non chargée (voir les logs)'))
                    print('⏱️  Générations longues (Hi-Res, SDXL) sans erreur 524 : ' + ('activé ✓' if jobs_ok else '⚠️ non disponible (voir les logs)'))
                    try:
                        _scripts = requests.get(f'http://127.0.0.1:{PORT}/sdapi/v1/scripts', timeout=5).json()
                        ad_ok = 'adetailer' in [str(x).lower() for x in _scripts.get('txt2img', [])]
                    except Exception:
                        ad_ok = False
                    print('🧑‍🎨 ADetailer (visages / mains) : ' + ('activé ✓' if ad_ok else '⚠️ non chargé (relancer la cellule 2, voir les logs)'))
                    print('💾 Sauvegarde des modèles (Hugging Face privé) : ' + ('activée ✓' if (_hf_token and _hf_repo) else 'désactivée (secrets HF_TOKEN / HF_REPO absents)'))
                    print('\n💡 Connexion à AbiArt :')
                    print('   1. Sur votre site : Backend "Kaggle" → collez l\'URL ci-dessus.')
                    print(f'   2. Ou dans Railway : Variables → KAGGLE_URL = {url}')
                    if ABIART_APP_URL.strip():
                        try:
                            r = requests.post(f"{ABIART_APP_URL.rstrip('/')}/api/kaggle/register", json={'url': url}, timeout=8)
                            print('\n✅ Connecté automatiquement à AbiArt !' if r.status_code == 200
                                  else f'\nℹ️ Auto-enregistrement : HTTP {r.status_code}')
                        except Exception:
                            pass
                    print('\n(Laissez cette cellule tourner : l\'arrêter coupe le backend.)\n')
                else:
                    print('\n⚠️ Tunnel Cloudflare introuvable. Logs :\n' + open(TLOG).read()[-1500:])
                print(f'🧠 Mémoire vive disponible : {ram_free_gb():.1f} Go')
                if USE_GPU2 and ram_free_gb() < 12:
                    # Charger un second modèle SDXL demande ~8 Go de RAM le temps du chargement
                    USE_GPU2 = False
                    print('⚠️ Second GPU non démarré : moins de 12 Go de mémoire vive disponibles. '
                          'Le backend fonctionne sur un seul GPU.\n')
                if USE_GPU2:
                    # Fichiers de réglages séparés : les deux instances n'écrivent pas dans le même config.json
                    proc2 = launch_webui(PORT2, 1, LOG2, ['--ui-settings-file', f'{BASE}/config_gpu1.json',
                                                           '--ui-config-file', f'{BASE}/ui-config_gpu1.json'])
                    t2 = time.time()
                    print('🚀 Second GPU : démarrage de la 2e instance (1 à 3 min). Le site est déjà utilisable.\n')
        # d) seconde instance (GPU 1) : facultative, un échec laisse le backend fonctionner sur un GPU
        if proc2 is not None and not ready2:
            full2 = open(LOG2, errors='replace').read() if os.path.exists(LOG2) else ''
            if proc2.poll() is not None:
                print(f'\n⚠️ Second GPU : la 2e instance s\'est arrêtée (code {proc2.returncode}). '
                      'Le backend continue sur un seul GPU. Fin du log :\n' + full2[-1500:])
                proc2 = None
            elif 'stable diffusion model failed to load' in full2.lower() or time.time() - t2 > 900:
                print('\n⚠️ Second GPU : la 2e instance n\'a pas démarré. Le backend continue sur un seul GPU. '
                      'Fin du log :\n' + full2[-1500:])
                try:
                    os.killpg(os.getpgid(proc2.pid), signal.SIGTERM)
                except Exception:
                    pass
                proc2 = None
            elif 'Model loaded in' in full2:
                try:
                    ready2 = requests.get(f'http://127.0.0.1:{PORT2}/sdapi/v1/options', timeout=5).status_code == 200
                except Exception:
                    ready2 = False
                if ready2:
                    print(f'\n✅ Second GPU actif ({time.time() - t2:.0f} s) : 2 générations peuvent tourner en même temps. '
                          f'Mémoire vive disponible : {ram_free_gb():.1f} Go\n')
        time.sleep(2)
except KeyboardInterrupt:
    print('\nArrêt demandé.')
finally:
    for p in (tunnel_proc, proc, proc2):
        try:
            if p and p.poll() is None:
                os.killpg(os.getpgid(p.pid), signal.SIGTERM)
        except Exception:
            pass